In [ ]:
from dotenv import dotenv_values
from requests import Session
from urllib.parse import urljoin
from pathlib import Path
from http import HTTPStatus
import xmltodict

# ROOT_DIR = Path(__file__).resolve().parent

ROOT_DIR = Path('.')
config: dict = dotenv_values(ROOT_DIR / ".env")
API_KEY = config["MEU_DANFE_API_KEY"]
BASE_URL = config.get("MEU_DANFE_API_URL", "")
headers = {"Api-Key": API_KEY, "accept": "application/json"}
"""
curl -X 'PUT' \
  'https://api.meudanfe.com.br/v2/fd/add/31240502068925000108550030001379331427845127' \
  -H 'accept: application/json' \
  -H 'Api-Key: b95580ef-eeea-4dd5-8722-91fa013e0794'
"""
# Source - https://stackoverflow.com/a/51026159
# Posted by rouble, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-25, License - CC BY-SA 4.0


class LiveServerSession(Session):
    def __init__(self, base_url="", api_key=None):
        super().__init__()
        self.base_url = base_url
        self.headers.update({"accept": "application/json", "Api-Key": api_key})

    def request(self, method, url, *args, **kwargs):
        joined_url = urljoin(self.base_url, url)
        return super().request(method, joined_url, *args, **kwargs)


session = LiveServerSession(base_url=BASE_URL, api_key=API_KEY)


def find_and_add_nfe(key: str, session) -> dict[str, str]:
    """
    URL Completa: https://api.meudanfe.com.br/v2/fd/add/{Chave-Acesso}

    A busca é feita simplesmente informando a Chave de Acesso no path {Chave-Acesso} e a Api-Key no header da requisição.
    A resposta é um objeto em JSON, contendo detalhes da solicitação com um dos seguintes status:
        WAITING: Na fila para consultar
        SEARCHING: Consultando ou aguardando resposta
        NOT_FOUND: Não encontrado ou não existe
        OK: Consulta realizada com sucesso
        ERROR: Falha ao consultar
    Busca de NF-e via chave de acesso é cobrado R$ 0,03 (3 centavos) por consulta.
    NÃO cobramos buscas de documentos já adicionadas no menu Minhas NFs / Minhas CTs
    Busca de qualquer CT-e já na nossa base de dados tambem são GRÁTIS. (NÃO fazemos busca de CT-e direto na Receita Federal)
    Após solicitação, o endpoint pode ser usado para consultar o status.
        AVISO: Varias solicitações da mesma chave de acesso em menos de 1 segundo BLOQUEARÁ sua conta e API Meu Danfe!
        AVISO: Para evitar o bloqueio, favor aguardar ao menos 1 segundo para consultar o status da solicitação.
    """

    try:
        response = session.request("PUT", f"fd/add/{key}")
        response.raise_for_status()
        return response
    except:
        None

def get_nfe(key: str, session) -> dict[str, str]:
    """
    URL Completa: https://api.meudanfe.com.br/v2/fd/get/xml/{Chave-Acesso}

    O download é feito simplesmente informando a Chave de Acesso no path {Chave-Acesso} e a Api-Key no header da requisição.
    A resposta é um objeto em JSON, contendo detalhes da solicitação e o XML em formato texto.
    """

    try:
        response = session.request("GET", f"fd/get/xml/{key}")
        return response
    except:
        None


def main(): ...


if __name__ == "__main__":
    main()


In [ ]:
"""
Série -> data['nfeProc']['NFe']['infNFe']['ide']['serie']
Data de Emissão -> data['nfeProc']['NFe']['infNFe']['ide']['dhEmi']
Data de Saída/Entrada -> data['nfeProc']['NFe']['infNFe']['ide']['dhSaiEnt']
CNPJ Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['CNPJ']
Razão Social Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['xNome']
UF Emitente -> data['nfeProc']['NFe']['infNFe']['emit']['enderEmit']['UF']
CNPJ Destinatário -> data['nfeProc']['NFe']['infNFe']['dest']['xNome']
Razão Social Destinatário -> data['nfeProc']['NFe']['infNFe']['dest']['CNPJ']
Natureza da Operação -> data['nfeProc']['NFe']['infNFe']['ide']['natOp']
Sequencial do Item -> data['nfeProc']['NFe']['infNFe']['det'][0]['nItem']
Código do Produto -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['cProd']
Descrição do Produto -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['xProd']
NCM -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['NCM']
CFOP -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['CFOP']
Unidade Comercial -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['uCom']
Quantidade -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['qCom']
Valor Unitário -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['vUnCom']
Valor Total Bruto do Item -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['vProd']
CST/CSOSN -> data['nfeProc']['NFe']['infNFe']['det'][0]['prod']['imposto']['ICMS']
Base de Cálculo ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Alíquota ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Valor ICMS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']['ICMS']
Base de Cálculo ICMS ST -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor ICMS ST -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor IPI -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Base de Cálculo PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Alíquotas PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valores PIS/COFINS -> data['nfeProc']['NFe']['infNFe']['det'][0]['imposto']
Valor Total da Nota -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vNF']
Valor Total dos Produtos -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vProd']
Frete -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vFrete']
Seguro -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vSeg']
Desconto -> data['nfeProc']['NFe']['infNFe']['total']['ICMSTot']['vDesc']
Informações Complementares -> data['nfeProc']['NFe']['infNFe']['infAdic']['infCpl']
"""

# Filter

In [71]:
import pandas as pd
import shutil
import re

todo = pd.read_csv('notas.tsv', sep='\t')
todo.PROJETO.unique()

WD = Path('/home/leonardo/Downloads/OneDrive_2026-04-25/02. Nacional/')
files = []
for item in WD.glob('*'):
    if item.is_dir() and item.name in todo.PROJETO.values:
        group = todo[todo.PROJETO == item.name]
        r = re.compile('|'.join(group.NOTAS.astype(str)))
        for file in item.rglob('**/*'):
            if re.search(r, file.name):
                files.append({'PROJECT': item.name, 'FILE': file})
todo.dtypes
found = pd.DataFrame(files)
found['NUMBER'] = found['FILE'].map(lambda x: re.search(r'\d{3,}', x.name).group())
found.query('PROJECT == "ANAGÉ"')
found['NUMBER'] = found['NUMBER'].astype(int)
todo.merge(found, how='left_anti', left_on=['PROJETO', 'NOTAS'], right_on=['PROJECT', 'NUMBER'])

import pymupdf
import re
from pathlib import Path

WD = Path('/home/leonardo/Downloads/OneDrive_2026-04-25/02. Nacional/')

def extract_key(file:Path):
    keys = []
    doc = pymupdf.open(file)
    for page in doc:
        page_text = page.get_text()
        matches = re.findall(r'\d(?:\s*\d){43}', page_text)
        keys.extend(re.sub(r'\s+', '', m) for m in matches)
    return keys
todo['CNPJ'] = todo['CNPJ'].str.replace(r'[^0-9]', '', regex=True)
chaves = pd.read_csv('chaves_nfe.csv', header=None, names=['chave'], dtype=str)
chaves[['cnpj', 'numero']] = chaves['chave'].str.extract(r'^\d{6}(\d{14})\d{5}(\d{9})', expand=True)
chaves['numero'] = chaves['numero'].astype(int)
todo.merge(
    chaves, how="inner", left_on=["CNPJ", "NOTAS"], right_on=["cnpj", "numero"]
).drop_duplicates()['chave'].to_csv('todo.csv', index=False)